# NB00 — Phase 0: audit before spending GPU time

**Budget: 1.5 h**

*Goal: settle everything for the project cost*

The paper's configuration was audited once by Claude against its code with file:line citations, in `se/paper_audit.json`. 
*TODO: Audit myself*

We do not train anything, just find the answers to the following questions to scope the project cost.

1. **Does the base repo's activation injection work at all?** 
   For the rotation experiment to work, we need the injected vector to actually reach the model. 
   In section 1 we see that the base repo's methods do not work and those runs must be discarded.
2. **What is the paper's own configuration?** 
   We load the frozen configuration from the paper found at `TransluceAI/introspective-interp` and 
   show they do not build a projector for the patching experiments. The projectors elsewhere are LoRA targets of rank 128.
3. Is the target architecture compatible with the transform (tying, quantization, dimensions)?
   In section 1, we show that
4. **What effect size is observable at this eval set size?** 
   Rotation can only destroy the value of `v`, which the paper's own `– activation` ablation bounds at about 4 percent. 
   Anything below the binomial CI half-width must not be preregistered as a reading, 
   and everything must be reported normalized to the no-activation floor .
5. **What is the numerical noise floor?** Phase 1's gate is calibrated against this number,
   not against an absolute.
6. Preregister the §9 readings, with numeric thresholds, before any results exist.

<!-- **Exit criteria (revised §4):** the reuse question settled *with the `target_modules` question
settled*, a stated eval `n` and CI, a measured KL noise floor, and the no-activation floor
established as the scale results are reported on.  -->

If the CI half-width under the ~4 percent activation contribution (Paper Table 5), enlarge the eval set before proceeding.

In [1]:
# --- dependencies -----------------------------------------------------------
# A RunPod image ships torch built against the pod's own driver, so nothing here may
# replace it: se_env pins the installed torch as a pip constraint and installs only what
# is missing or below the floor these notebooks need — including bitsandbytes, which Colab
# had preinstalled and RunPod images do not (se_common asks for paged_adamw_8bit).
# Safe to re-run: a no-op on a warm pod.
import os
import sys

# `se/` holds the shared modules: se_env, se_config, rotate, se_common. Clone this repo
# onto the pod's volume (/workspace/self_explainer) so it survives the pod.
REPO_DIR = os.environ.get("SE_REPO_DIR", "/workspace/self_explainer")
if not os.path.isdir(os.path.join(REPO_DIR, "se")):
    REPO_DIR = os.path.abspath(".." if os.path.isdir("../se") else ".")
sys.path.insert(0, os.path.join(REPO_DIR, "se"))

import se_env

se_env.ensure_deps()


dependencies: already satisfied (torch 2.8.0+cu128, transformers 5.15.0, peft 0.20.0, trl 1.10.0, bitsandbytes 0.50.1)


{}

In [2]:
# --- environment ------------------------------------------------------------
# Caches and outputs go on the pod's volume, never the container disk: /workspace is what
# survives a stopped or terminated pod, and the 8B checkpoint alone is 16 GB. HF_TOKEN comes
# from the pod template's environment or <volume>/.hf_token — there is no prompt to answer,
# because a preempted pod restarts with nobody watching.
#
# Only section 7 (the KL noise floor) touches the GPU, so a CPU pod is enough for the rest.
env = se_env.bootstrap(repo_dir=REPO_DIR, gpu="optional", min_vram_gb=24)

import se_config as C


RUNPOD ENVIRONMENT
host    : RunPod pod b6z487qaraq9cw  |  python 3.12.3
gpu     : 1 x NVIDIA A100 80GB PCIe  |  79 GiB  |  bf16 yes
torch   : 2.8.0+cu128  (CUDA 12.8)
stack   : transformers 5.15.0 · peft 0.20.0 · trl 1.10.0 · datasets 5.0.1 · accelerate 1.14.0 · bitsandbytes 0.50.1
repo    : /workspace/self_explainer
volume  : /workspace  (own mount, 637201 GiB free)
hf cache: /workspace/.cache/huggingface/
outputs : /workspace/self_explainer
hf token: none
!! no HF token found. Public checkpoints still work; set HF_TOKEN in the pod template or write it to /workspace/.hf_token if a download 401s.


## 1. Does the injected activation reach the model?

We decide that we cannot reuse the run data from the `introspection-colab` repo. 
The base repo's activation-patching path builds its input embeddings like this
(`finished_notebooks/no_quantization/notebook_Qwen3_8B_no_quant.ipynb`,
`build_inputs_embeds_projected` — the same nine lines appear in every finished notebook):

```python
input_embeds = model.get_input_embeddings()(input_ids)
mask = (input_ids == placeholder_id).unsqueeze(-1)
v = projection(intervention_vectors, chunk_ids)
input_embeds.masked_scatter(mask, v.to(input_embeds.dtype))   # <-- result discarded
return input_embeds
```

`Tensor.masked_scatter` is out-of-place. The return value is dropped, so `input_embeds` is never modified.
The projected activation never enters the forward pass, and the projection never receives a gradient.
The rotation experiment would result in a flat null. We reproduce the code below to test.

In [3]:
import torch

import mini_qwen3
from se_common import ChunkInputMap, build_inputs_embeds

d = 32
model = mini_qwen3.build(d=d, seed=1)
placeholder_id = 7

ids = torch.randint(0, 256, (3, 12))
ids[:, 5] = placeholder_id
vectors = torch.randn(3, d, dtype=torch.float64) * 5.0
chunk_ids = torch.zeros(3, dtype=torch.long)
imap = ChunkInputMap(d, d, 1, "full", base="identity").double()

plain = model.get_input_embeddings()(ids)


def base_repo_injection(model, input_map, input_ids, vectors, chunk_ids, placeholder_id):
    """Verbatim reproduction of the base repo's build_inputs_embeds_projected."""
    e = model.get_input_embeddings()(input_ids)
    mask = (input_ids == placeholder_id).unsqueeze(-1)
    v = input_map(vectors, chunk_ids)
    e.masked_scatter(mask, v.to(e.dtype))
    return e


bugged = base_repo_injection(model, imap, ids, vectors, chunk_ids, placeholder_id)
fixed = build_inputs_embeds(model, imap, ids, vectors, chunk_ids, placeholder_id)

injection_is_noop = torch.equal(bugged, plain)
print(f"base repo injection is a no-op : {injection_is_noop}")
print(f"fixed injection changes embeds : {not torch.equal(fixed, plain)}")
print(f"placeholder holds exactly Pi v : {(fixed[:, 5] - vectors).abs().max().item():.2e}")
print(f"other positions untouched      : "
      f"{torch.equal(fixed[:, [0,1,2,3,4,6,7]], plain[:, [0,1,2,3,4,6,7]])}")

assert injection_is_noop, (
    "The base repo's injection is NOT a no-op in this environment — the finding above does "
    "not hold, so neither does the reuse verdict below: the finished runs may contain a "
    "real identity control. Re-read before proceeding."
)


base repo injection is a no-op : True
fixed injection changes embeds : True
placeholder holds exactly Pi v : 0.00e+00
other positions untouched      : True


**Nothing from the previous project is reusable.** 
The patching runs were trained without the activation, so none of them is relevant to our project.

## 2. What is the paper's configuration actually?

The findings from the audit of `TransluceAI/introspective-interp` decide the arm matrix, so
they are carried as data — `se/paper_audit.json`, one entry per finding with its file:line
citations and the arm it sets — rather than as prose to be recalled. The audit was run once
against a checkout; freezing it means the pod needs no clone of the release, and a reader can
check any line of it against the public repo. `paper_audit.json` lists the grep commands that
re-run it if you ever want to.

**F.1 — the patching task uses no projector, on either side.** All four `config/act_patch/*.yaml`
omit `use_embed_proj`; `train.py:76` reads `config.get("use_embed_proj", False)` and passes it
explicitly, overriding `ContinuousQwen`'s class default of `True`; and `continuous_base.py:120`
only builds projectors when the flag is set *or* the dimensions differ — Qwen3-8B and
Llama-3.1-8B are both 4096. So `embed_projs = None`, the activation is injected raw, and the
paper's Tables 2 and 5 are *symmetric*. Two consequences: any critique alleging projector
asymmetry is wrong, and **`C0` is the paper's configuration, not a strawman rung.** It cannot
represent `Q^{-1}`, and that is a fact about the paper's setup rather than an artifact of ours.

**F.3 — projectors are LoRA target modules.** `model/utils.py:252–253` appends
`embed_projs.{i}` to `target_modules` for every trainable projector, and the configs set
`lora_r: 128`. A "full-rank" projector is therefore frozen-at-init plus a rank-128 update; the
paper's footnote 7 says so and is accurate. At `d = 4096` that cannot approximate `Q^T`. If our
`Cfull` ever lands inside `target_modules`, the central arm silently becomes `C128` and the
experiment measures LoRA rank — which is why `se_common` asserts the parameter count on every
run and `se/test_input_map.py` proves the assertion can fail.

**F.2 — a pretrained Qwen projector exists.** `config/feature_descriptions/qwen_131k.yaml` sets
`use_embed_proj: true` and loads
`alignment_outputs/qwen_llama_3.1_8b_base/final_alignment_model.pt`, writing to
`checkpoints/aligned_qwen_pretrained`; the random-init condition is a separate config writing to
`nonaligned_qwen`. The paper's §3.1 says pretrained projections were included only for
Llama-3.1-70B. So the most direct test of basis compatibility — same explainer, same target,
same data, alignment as the only variable — was configured and never reported. NB05 §7.4 turns
that into the `P-rand` vs `P-rand-full` arm.

The cell below prints that record and then checks the part that is *ours*: that the arms the
findings imply actually exist in `se_config`, and that our input map is not routed through
PEFT. Those three assertions are the only thing here that can fail.

In [4]:
import json

import se_common as S

# The frozen audit (se/paper_audit.json). Read from this repo, not from a checkout of the
# release: the findings are file:line citations, so freezing them costs nothing but a clone.
with open(C.PAPER_AUDIT_PATH) as f:
    paper_audit = json.load(f)

print(f"{paper_audit['source']}, audited {paper_audit['audited_on']}")
print(f"record of audit: {paper_audit['audit_of_record']}\n")
for finding in paper_audit["findings"]:
    print(f"  {finding['id']}  {finding['claim']}")
    print(f"        sets arm: {finding['sets_arm']}")
    for line in finding["evidence"]:
        print(f"        - {line}")
    print()

# --- our own side of the audit (revised section 4.1) -------------------------
# This is the half that can fail: the findings above are fixed, but whether se_config still
# encodes them is a live question every time someone edits it.
print("our own configuration:")
own = [
    ("input map is outside LoRA target_modules",
     "input_map" not in C.LORA_TARGET_MODULES and "embed_projs" not in C.LORA_TARGET_MODULES),
    ("C128 exists as an explicit arm at the paper's rank",
     C.CAPACITY_RANK.get("C128") == C.PAPER_LORA_R),
    ("C0 is labelled as the paper's configuration",
     "paper" in C.CAPACITY_STATUS["C0"]),
]
for name, ok in own:
    print(f"  [{'ok  ' if ok else 'FAIL'}] {name}")
    assert ok, name

d_map = 4096
print(f"\n  Cfull trains {S.expected_input_map_params('Cfull', d_map, d_map)/1e6:.1f}M parameters "
      f"({C.N_LAYER_CHUNKS} chunks x {d_map}^2)")
print(f"  C128  trains {S.expected_input_map_params('C128', d_map, d_map)/1e6:.1f}M — "
      f"{100 * S.expected_input_map_params('C128', d_map, d_map) / S.expected_input_map_params('Cfull', d_map, d_map):.1f}% "
      f"of Cfull, and it cannot represent an arbitrary orthogonal map")
print(f"  the paper's per-LAYER projector would be "
      f"{S.expected_input_map_params('Cfull', d_map, d_map, 32)/1e6:.0f}M "
      f"(section 3.1's 537M); ours is per layer CHUNK")

# Copied onto the volume so the run's report directory is self-contained: NB07 quotes these
# findings, and a reader of the outputs should not have to open the source tree.
with open(f"{C.REPORTS_DIR}/code_audit.json", "w") as f:
    json.dump(paper_audit, f, indent=2)
print(f"\nwritten to {C.REPORTS_DIR}/code_audit.json — Appendix F is a deliverable (section 11)")

TransluceAI/introspective-interp @ main, audited 2026-08-16
record of audit: claude_plans/v2_revisions_claude.md, Appendix F

  F.1  The activation-patching task uses no projector, on either side.
        sets arm: C0
        - config/act_patch/{base_base,base_qwen,qwen_base,qwen_qwen}_act_patch_cf.yaml — all four omit use_embed_proj
        - train.py:76 — use_embed_proj=config.get("use_embed_proj", False)
        - model/utils.py:116 — passes that value explicitly, overriding ContinuousQwen's class default of True (model/continuous_qwen.py:25; ContinuousLlama defaults False at model/continuous_llama.py:22)
        - model/continuous_base.py:120 — builds projectors only if use_embed_proj or subject_embed_dim != hidden_size; Qwen3-8B and Llama-3.1-8B are both 4096, so the fallback never fires

  F.2  A pretrained Qwen projector exists; the paper says it does not.
        sets arm: P-rand vs P-ridge (NB05, §7.4)
        - config/feature_descriptions/qwen_131k.yaml — use_embed_proj: true

## 3. Architecture compatibility

Checking that `se_config` has the right architecture. We need to ensure no quantization so $Q$-invariance holds.


In [5]:
from transformers import AutoConfig

target_cfg = AutoConfig.from_pretrained(C.TARGET_MODEL_ID)
print(f"{C.TARGET_MODEL_ID}")
print(f"  hidden_size          : {target_cfg.hidden_size}")
print(f"  num_hidden_layers    : {target_cfg.num_hidden_layers}")
print(f"  num_attention_heads  : {target_cfg.num_attention_heads}")
print(f"  num_key_value_heads  : {target_cfg.num_key_value_heads}  (GQA: k/v have fewer output "
      f"heads but full input dim, so W Q^T is well-formed)")
print(f"  head_dim             : {getattr(target_cfg, 'head_dim', None)}")
print(f"  tie_word_embeddings  : {target_cfg.tie_word_embeddings}")
print(f"  rms_norm_eps         : {target_cfg.rms_norm_eps}")

assert not target_cfg.tie_word_embeddings, (
    "Target has tied embeddings. Folding model.norm's gain into lm_head would corrupt the "
    "embedding matrix, since they are the same tensor. Use rotate.untie_embeddings() first "
    "(costs one extra [vocab, d] tensor) or keep 8B as the target."
)
assert not C.USE_4BIT, "Invariance is meaningless in 4-bit (§5.3)."
print("\ncompatible.")


Qwen/Qwen3-8B
  hidden_size          : 4096
  num_hidden_layers    : 36
  num_attention_heads  : 32
  num_key_value_heads  : 8  (GQA: k/v have fewer output heads but full input dim, so W Q^T is well-formed)
  head_dim             : 128
  tie_word_embeddings  : False
  rms_norm_eps         : 1e-06

compatible.


## 4. Dataset schema

The second gate of NB01 needs the token position to send the patch through the hook path. 
Ensuring the schema is correct now. 


In [6]:
from datasets import load_dataset

probe = load_dataset(C.ACT_DATASET, split="train[:4]")
ex = probe[0]

print("top-level fields:")
for k, v in ex.items():
    shape = f"list[{len(v)}]" if isinstance(v, list) else type(v).__name__
    print(f"  {k:>24} : {shape}")

print("\npatch_position fields:")
for k, v in ex["patch_position"].items():
    shape = f"list[{len(v)}]" if isinstance(v, list) else repr(v)[:60]
    print(f"  {k:>24} : {shape}")

vec_dim = len(ex["patch_position"]["intervention_vector"])
print(f"\nintervention_vector dim : {vec_dim}")
assert vec_dim == target_cfg.hidden_size, (
    f"cached vectors are {vec_dim}-dim but the target is {target_cfg.hidden_size}-dim"
)
print(f"layer field example     : {ex['layer']}")
print(f"is_different            : {ex['is_different']}")


top-level fields:
                     layer : list[9]
              input_tokens : list[40]
     original_continuation : list[1]
      ablated_continuation : list[1]
              is_different : bool
            patch_position : dict
       counterfactual_text : str
        gt_original_target : str
  gt_counterfactual_target : str
            layer_hashable : list[9]
                token_type : str
         __index_level_0__ : int

patch_position fields:
           counterfact_pos : 36
    counterfact_text_token : 'ĊĊ'
       intervention_vector : list[4096]
                  orig_pos : 37
           orig_text_token : 'ĊĊ'

intervention_vector dim : 4096
layer field example     : [27, 28, 29, 30, 31, 32, 33, 34, 35]
is_different            : False


## 5. What does the capacity ladder cost?

We check shapes and sizes for each trained projection at different steps on the capacity ladder.
The full-rank input map should be $4096 \times 4096 \times 4$ parameters. 


In [7]:
import pandas as pd

import se_common as S

cfg = target_cfg
d = cfg.hidden_size
ff = cfg.intermediate_size
head_dim = getattr(cfg, "head_dim", d // cfg.num_attention_heads)
kv_dim = cfg.num_key_value_heads * head_dim
q_dim = cfg.num_attention_heads * head_dim

shapes = {                          # (in_features, out_features) per LoRA target module
    "q_proj": (d, q_dim), "k_proj": (d, kv_dim), "v_proj": (d, kv_dim),
    "o_proj": (q_dim, d), "gate_proj": (d, ff), "up_proj": (d, ff), "down_proj": (ff, d),
}
lora_params = cfg.num_hidden_layers * sum(
    C.LORA_R * (i + o) for name, (i, o) in shapes.items() if name in C.LORA_TARGET_MODULES
)

REPRESENTS = {"C0": "no", "C8": "no", "C128": "no", "C512": "partially",
              "Cfull": "exactly", "Cfull-rand": "exactly", "Oracle": "exactly (frozen)"}

rows = []
for cap in C.CAPACITIES:
    total = S.expected_input_map_params(cap, d, d)
    rows.append({
        "arm": cap, "rank": C.CAPACITY_RANK[cap],
        "params_per_chunk": total // C.N_LAYER_CHUNKS,
        f"params_x{C.N_LAYER_CHUNKS}_chunks": total,
        "pct_of_lora_budget": 100 * total / lora_params,
        "can_represent_Q_inv": REPRESENTS[cap],
        "status": C.CAPACITY_STATUS[cap],
    })

print(f"LoRA trainable params (r={C.LORA_R}, {len(C.LORA_TARGET_MODULES)} modules x "
      f"{cfg.num_hidden_layers} layers): {lora_params/1e6:.1f}M")
print(f"input map is trained whole, never through LoRA: "
      f"{'input_map' not in C.LORA_TARGET_MODULES and 'embed_projs' not in C.LORA_TARGET_MODULES}"
      f"  (§3.1's trap box; se_common asserts this on every run)\n")
budget = pd.DataFrame(rows)
budget


LoRA trainable params (r=16, 7 modules x 36 layers): 43.6M
input map is trained whole, never through LoRA: True  (§3.1's trap box; se_common asserts this on every run)



,arm,rank,params_per_chunk,params_x4_chunks,pct_of_lora_budget,can_represent_Q_inv,status
0,C0,0,0,0,0.000000,no,the paper's configuration (no map at all)
1,C8,8,65536,262144,0.600601,no,ladder interior
2,C128,128,1048576,4194304,9.609610,no,the paper's rank cap on a projector (F.3)
3,C512,512,4194304,16777216,38.438438,partially,ladder interior
4,Cfull,full,16777216,67108864,153.753754,exactly,primary augmented arm
5,Cfull-rand,full,16777216,67108864,153.753754,exactly,init control
6,Oracle,0,0,0,0.000000,exactly (frozen),representability ceiling


## 6. What effect size is observable, against what range?

We check how much evidence the sweep can actually provide. 
Rotation can only destroy the value of `v`; it cannot degrade what the explainer gets from
the prompt. 
The paper's `– activation` ablation (Table 5) bounds that value at ~4 points using the exact match for Qwen3-8B self-explanation 
We compare this to a self-non-self margin of of ~10. 
So the entire measurable range of the rotation arm is ~4 points, meaning all results will be reported normalized to this floor.
To get more statistical power, we increase `EVAL_SIZE = 1024` (from 128 in the replication).


In [8]:
import math

n_eval = C.EVAL_SIZE


def binomial_half_width(p, n, z=1.96):
    return z * math.sqrt(p * (1 - p) / n)


print(f"eval set size n = {n_eval}\n")
print(f"{'true score':>12} {'95% CI half-width':>20}")
for p in (0.3, 0.5, 0.64, 0.8):
    print(f"{p:>12.2f} {binomial_half_width(p, n_eval):>20.4f}")

worst = binomial_half_width(0.5, n_eval)
print(f"\nworst case (p=0.5): +/- {worst:.4f}")
print(f"difference of two independent arms: +/- {worst * math.sqrt(2):.4f}")

# --- the task's own dynamic range (revised §4.2) ---------------------------
# Rotation can only destroy the value of the injected vector. The paper's `- activation`
# ablation is what bounds that value, so it, not the self-vs-cross margin, is the scale.
activation_worth = C.PAPER_WITH_ACTIVATION - C.PAPER_NO_ACTIVATION
print(f"\nthe activation is worth (paper's Table 5, Qwen3-8B self):")
print(f"  with activation      : {C.PAPER_WITH_ACTIVATION:.3f}")
print(f"  without activation   : {C.PAPER_NO_ACTIVATION:.3f}   <- the floor")
print(f"  entire dynamic range : {activation_worth:.3f}  "
      f"({100 * activation_worth:.1f} points)")
print(f"  self-vs-cross margin : {C.PAPER_SELF_CROSS_MARGIN:.3f}  "
      f"(not our range: rotation cannot touch the prompt)")

resolvable = worst < activation_worth
print(f"\nhalf-width {worst:.4f} vs range {activation_worth:.4f}: "
      f"{'resolvable' if resolvable else 'NOT RESOLVABLE'}")
print(f"a 1-point raw gap is {100 / (100 * activation_worth):.0f}% of the range; "
      f"the eval resolves ~{100 * worst / activation_worth:.0f}% of it")

for target in (0.04, 0.02):
    need = math.ceil((1.96 ** 2) * 0.25 / target ** 2)
    print(f"  n for a +/-{target:.2f} half-width at p=0.5: {need}")

if not resolvable:
    print("\nSTOP: the eval set cannot resolve the activation's entire contribution. Enlarge")
    print("EVAL_SIZE — revised §4.2's exit criterion — before preregistering anything.")
else:
    print("\nEVAL_SIZE clears the bar on the independent CI. The readings are stated on")
    print("PAIRED differences anyway (same eval items across arms), which is several times")
    print("tighter: se = sqrt(b + c) / n over the discordant pairs. se_common.paired_delta")
    print("computes it from eval_records.json, and NB03 reports it beside every gap.")

# the paper's Table 2 self-margins, kept for context: they are what the *task* separates,
# not what rotation can move
margins = {"patching (Qwen3)": 0.640 - 0.541, "input ablation (Qwen3)": 0.834 - 0.581}
print("\npaper's Table 2 self-margins, for context:")
for k, v in margins.items():
    print(f"  {k:>24}: {v:.3f}  "
          f"{'detectable' if v > worst * math.sqrt(2) else 'BELOW RESOLUTION'}")


eval set size n = 1024

  true score    95% CI half-width
        0.30               0.0281
        0.50               0.0306
        0.64               0.0294
        0.80               0.0245

worst case (p=0.5): +/- 0.0306
difference of two independent arms: +/- 0.0433

the activation is worth (paper's Table 5, Qwen3-8B self):
  with activation      : 0.640
  without activation   : 0.599   <- the floor
  entire dynamic range : 0.041  (4.1 points)
  self-vs-cross margin : 0.099  (not our range: rotation cannot touch the prompt)

half-width 0.0306 vs range 0.0410: resolvable
a 1-point raw gap is 24% of the range; the eval resolves ~75% of it
  n for a +/-0.04 half-width at p=0.5: 601
  n for a +/-0.02 half-width at p=0.5: 2401

EVAL_SIZE clears the bar on the independent CI. The readings are stated on
PAIRED differences anyway (same eval items across arms), which is several times
tighter: se = sqrt(b + c) / n over the discordant pairs. se_common.paired_delta
computes it from eval_reco

## 7. The numerical noise floor

We run $M$ twice on the same 256 sequences at different batch sizes, report mean per-token
KL. We gate Phase 1's using this number.

This needs the 8B model, so it is the one cell here that wants a GPU: it runs itself on a GPU
pod and skips itself on a CPU pod, rather than depending on a flag someone remembers to flip.
It is also the cell that decides whether NB01's threshold means anything, so if it skips here,
run it at the start of NB01 before reading the gate.


In [9]:
RUN_NOISE_FLOOR = env["has_gpu"]      # a pod either has a GPU or it does not

if RUN_NOISE_FLOOR:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    import rotate as R

    tok = AutoTokenizer.from_pretrained(C.TARGET_MODEL_ID)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(
        C.TARGET_MODEL_ID, dtype=torch.bfloat16, device_map="auto").eval()

    fw = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train",
                      streaming=True)
    noise_texts = []
    for row in fw:
        if len(row["text"]) > 2000:
            noise_texts.append(row["text"])
        if len(noise_texts) >= 64:
            break

    # same model, same inputs, different batch size -> pure arithmetic noise
    floor = R.invariance_gate(m, m, tok, noise_texts, max_length=512, batch_size=1)
    floor_b4 = R.invariance_gate(m, m, tok, noise_texts, max_length=512, batch_size=4)
    print(f"batch 1 vs itself : mean KL {floor['mean_kl']:.3e}")
    print(f"batch 4 vs itself : mean KL {floor_b4['mean_kl']:.3e}")
    print("\nNB: invariance_gate runs both models batch-identically, so this measures")
    print("within-batching noise. For the cross-batching floor, compare logits collected")
    print("at batch 1 against batch 4 for the same sequences.")

    with open(f"{C.REPORTS_DIR}/noise_floor.json", "w") as f:
        json.dump({"batch1": floor, "batch4": floor_b4}, f, indent=2)
else:
    print("skipped — no GPU visible on this pod. Run this section on a GPU pod, or at "
          "the start of NB01, before reading its gate against the 1e-4 threshold.")


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

batch 1 vs itself : mean KL 0.000e+00
batch 4 vs itself : mean KL 0.000e+00

NB: invariance_gate runs both models batch-identically, so this measures
within-batching noise. For the cross-batching floor, compare logits collected
at batch 1 against batch 4 for the same sequences.


## 8. Preregistration

A table of readings is only a preregistration if the mapping from numbers to readings is fixed in advance. 
The cell below writes the preregistration table and the quantitative thresholds NB07 will apply, and timestamps it.
Preregistration was written by Claude and type checked by me.

In [10]:
import datetime

# Version 2: written after the audit of TransluceAI/introspective-interp (§1b above). Version 1
# predates it, stated its thresholds on the raw score scale, and had no row for the rank-artifact
# hypothesis. If a version-1 file exists on disk this cell refuses to silently replace it —
# rewriting a preregistration after seeing results is the one thing it exists to prevent.
PREREG_VERSION = 2

PREREGISTRATION = {
    "version": PREREG_VERSION,
    "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "primary_metric": "exact_match",
    "secondary_metrics": ["has_changed_f1", "content_match"],
    "eval_n": C.EVAL_SIZE,
    "binomial_half_width_at_p50": binomial_half_width(0.5, C.EVAL_SIZE),

    # §4.2: the scale everything is reported on. Rotation can only destroy the value of v, so
    # the denominator is the activation's contribution, not the score.
    "reporting_scale": "normalized to the no-activation floor",
    "normalization": {
        "retained": "(score(arm) - floor) / (score(Cfull, R-id) - floor), at matched N",
        "destroyed": "1 - retained",
        "floor": "the v=0 arm measured in NB03 at the same N, NOT the paper's 59.9",
    },
    "activation_range_reference": {
        "with_activation": C.PAPER_WITH_ACTIVATION,
        "without_activation": C.PAPER_NO_ACTIVATION,
        "points": round(100 * (C.PAPER_WITH_ACTIVATION - C.PAPER_NO_ACTIVATION), 1),
        "source": "paper Table 5, Qwen3-8B self-explanation on activation patching",
        "note": "context only; the floor used in every reading is our own measurement",
    },

    "thresholds": {
        # normalized, i.e. fractions of the activation's own contribution
        "overlap_destroyed": 0.15,
        "separated_destroyed": 0.35,
        "floor_match_retained": 0.10,
        "below_floor_retained": -0.10,
        "recovery_fraction": 0.90,
        "rank_artifact_closure": 0.50,
        # raw score points, for the comparisons that are levels rather than ranges
        "baseline_match_gap": 0.02,
        "overlap_gap_raw": 0.02,
        "separated_gap_raw": 0.05,
        "ridge_identity_tol": 1e-2,
        "exactness_tol": 0.02,
        "use_seed_band": True,        # "separated" also requires the seed bands not to overlap
    },

    "readings": [
        # --- primary: does coordinate correspondence matter? ----------------------
        {"id": "no_effect", "group": "primary",
         "observation": "Cfull curves for R-id and R-Q overlap at all N, both above the "
                        "no-activation floor",
         "rule": "abs(destroyed) < overlap_destroyed for every N, and both arms clear the floor",
         "reading": "Coordinate alignment contributes nothing recoverable; the paper's §3.4 "
                    "correlation is confounded"},
        {"id": "sample_efficiency", "group": "primary",
         "observation": "R-Q separated at low N, converging by the largest N",
         "rule": "destroyed(min N) > separated_destroyed and abs(destroyed(max N)) < "
                 "overlap_destroyed",
         "reading": "Coordinate frame is a sample-efficiency effect — reframes the paper's §4 "
                    "data-efficiency claim as the real finding",
         "prior": "modal outcome"},
        {"id": "attainable_optimum", "group": "primary",
         "observation": "R-Q strictly below R-id at all N, gap larger than the seed band",
         "rule": "destroyed(N) > max(separated_destroyed, seed band) for every N",
         "reading": "Frame affects the attainable optimum; strongest form of the critique"},
        {"id": "collapse_to_floor", "group": "primary",
         "observation": "R-Q falls to the no-activation floor",
         "rule": "retained(R-Q) < floor_match_retained at the largest N",
         "reading": "The activation's ENTIRE contribution was coordinate-frame"},
        {"id": "below_floor", "group": "primary",
         "observation": "R-Q falls below the no-activation floor",
         "rule": "retained(R-Q) < below_floor_retained",
         "reading": "Bug, or the rotated vector is actively misleading the explainer — "
                    "investigate before reporting anything"},
        {"id": "collapse_to_baseline", "group": "primary",
         "observation": "R-Q collapses to the cross-family baseline (E_cross, P-rand)",
         "rule": "abs(score_Q - score_cross_model) < baseline_match_gap at the largest N",
         "reading": "Table 1's ordering is a statement about geometry, not selfhood"},
        {"id": "all_inside_band", "group": "primary",
         "observation": "Every arm sits inside the seed band",
         "rule": "max destroyed across N < seed band width",
         "reading": "The effect is smaller than run-to-run variance at this scale. Report the N "
                    "at which it would become detectable — a real sensitivity result, not a hedge"},
        {"id": "metrics_disagree", "group": "primary",
         "observation": "The three metrics point in different directions",
         "rule": "sign(gap) differs across exact_match / has_changed_f1 / content_match",
         "reading": "Report all three separately and do not aggregate; state which component of "
                    "the task the frame affects"},

        # --- capacity: is the collapse about basis or about rank? -----------------
        {"id": "capacity_collapse", "group": "capacity",
         "observation": "C0 and C128 collapse under R-Q while Cfull does not",
         "rule": "retained(Q, C0) < floor_match_retained and retained(Q, Cfull) - "
                 "retained(Q, C0) > separated_destroyed",
         "reading": "Expected, and it is a capacity result, not evidence about selfhood: the "
                    "paper's own configuration cannot represent Q^-1 (Appendix F.1). Label it "
                    "that way in the figure"},
        {"id": "oracle_gap", "group": "capacity",
         "observation": "Oracle >> Cfull/R-Q at the largest ladder N",
         "rule": "retained(Q, Oracle) - retained(Q, Cfull) > separated_destroyed",
         "reading": "The map is representable but not learnable at this N; the ladder is "
                    "measuring optimization, not basis"},
        {"id": "low_rank_recovery", "group": "capacity",
         "observation": "Recovery threshold sits at low rank (C8)",
         "rule": "min rank r with retained(Q, Cr) >= recovery_fraction is <= 8",
         "reading": "Frame mismatch is low-dimensional; suggests a very cheap alignment recipe"},
        {"id": "cfull_rand_matches", "group": "capacity",
         "observation": "Cfull-rand gap approximately equals the Cfull gap",
         "rule": "abs(destroyed_rand - destroyed_identity_init) < overlap_destroyed at the "
                 "ladder N",
         "reading": "The R-id initialization head start was not driving the result"},

        # --- constructive: does the alternative recipe work? ----------------------
        {"id": "ridge_frozen_works", "group": "constructive",
         "observation": "P-ridge-frozen matches or approaches E_self",
         "rule": "abs(score_ridge_frozen - score_self) < overlap_gap_raw at the largest N",
         "reading": "The alternative recipe works with NO per-target training; primary practical "
                    "result"},
        {"id": "rank_artifact", "group": "constructive",
         "observation": "P-rand-full >> P-rand, approaching P-ridge",
         "rule": "(score_rand_full - score_rand) > rank_artifact_closure * "
                 "(score_ridge - score_rand)",
         "reading": "The paper's random-vs-pretrained projector gap is largely a rank-128 "
                    "artifact rather than evidence about activation alignment. Reinterprets the "
                    "mechanism in its §3.4, and the 70B numbers that rest on it become a "
                    "measurement of rank capacity"},
        {"id": "rank_not_binding", "group": "constructive",
         "observation": "P-rand-full approximately equals P-rand",
         "rule": "abs(score_rand_full - score_rand) < overlap_gap_raw",
         "reading": "Rank is not the binding constraint; initialization quality genuinely "
                    "reflects alignment, and the paper's reading stands"},
        {"id": "ridge_works", "group": "constructive",
         "observation": "Ridge-init cross-model explainer matches the self-explainer",
         "rule": "abs(score_ridge_cross - score_self) < overlap_gap_raw at the largest N",
         "reading": "A good initialization helps, but this is still a per-target training run — "
                    "weaker than ridge_frozen"},

        # --- integrity: stop conditions ------------------------------------------
        {"id": "ridge_identity_broken", "group": "integrity",
         "observation": "||Pi^Q - Pi Q^T||_F is not approximately 0",
         "rule": "max relative error > ridge_identity_tol",
         "reading": "Bug in the ridge implementation. Stop."},
        {"id": "exactness_failed", "group": "integrity",
         "observation": "Oracle/R-Q differs from C0/R-id beyond bf16 noise",
         "rule": "any metric delta > exactness_tol",
         "reading": "Plumbing bug. Stop — everything downstream is contaminated."},
        {"id": "full_rank_violated", "group": "integrity",
         "observation": "A Cfull run's input map was LoRA-wrapped or short on parameters",
         "rule": "any run's input_map_lora_wrapped is non-empty, or trainable != expected",
         "reading": "The central arm measured LoRA rank rather than basis (Appendix F.3). Stop, "
                    "fix target_modules, rerun."},
        {"id": "control_moved", "group": "integrity",
         "observation": "Input-ablation control moves beyond the seed band",
         "rule": "ablation score delta across rotation labels > seed band width",
         "reading": "Bug. Stop."},
        {"id": "prereg_compromised", "group": "integrity", "manual": True,
         "observation": "Any R-id cell was inspected before this file was written",
         "rule": "recorded by hand — no code can check it",
         "reading": "Preregistration is compromised for that cell; say so explicitly in the "
                    "write-up"},
    ],

    "not_mutually_exclusive": True,
    "compromised_cells": "None: no R-id results from finished_notebooks/ are reused, because "
                         "the injection bug voids them. Had they been reused, "
                         "preregistration would be partially compromised for those cells.",
    "notes": [
        "C0 is the paper's own patching configuration, not a strawman rung: all four act_patch "
        "configs omit use_embed_proj, train.py defaults it to False, and both models are "
        "4096-dim so the dimension-mismatch fallback never fires (Appendix F.1). Cfull is an "
        "augmentation introduced so the rotation question is answerable; label both.",
        "Projectors in the paper are LoRA target modules at rank 128 (Appendix F.3), so C128 is "
        "the paper's real capacity and also the failure mode of a plausible implementation of "
        "Cfull. Every run asserts its input-map parameter count for that reason.",
        "P-rand is the paper's cross-model condition faithfully: random dense init correctable "
        "only by a rank-128 update. P-rand-full lifts the cap and changes nothing else.",
        "Identity arm is rerun rather than reused: the base repo's activation injection was a "
        "no-op (NB00 section 2), so its finished patching runs are not a valid control.",
        "The Qscaled arm is a data-side transform only: a diagonal scaling breaks RMSNorm "
        "invariance, so there is no model whose activations are S v and whose function is "
        "unchanged. Its result speaks to distributional compatibility, not to invariance.",
        "The explainer is initialized from UNROTATED M and fed activations from M_Q. Building it "
        "from M_Q would rotate the explainer too, preserving alignment exactly and making the "
        "whole experiment a no-op.",
        "Cfull under R-id is initialized AT its solution, so low-N separation is partly an "
        "initialization artifact. Cfull-rand and Oracle exist to bound that.",
        "The ladder's honest reading is narrower than 'the minimum recovering rank measures the "
        "coordinate-frame share': at N=128 a 4096^2 map is being fit from ~128 vectors, so low "
        "ranks win at low N for bias-variance reasons with the basis effect inside that. Oracle "
        "plus the ladder is what disentangles them (§3.5).",
    ],
}

path = f"{C.REPORTS_DIR}/preregistration.json"
existing = json.load(open(path)) if os.path.exists(path) else None

if existing and existing.get("version") == PREREG_VERSION:
    print(f"preregistration v{PREREG_VERSION} already exists at {path} — not overwriting.")
    PREREGISTRATION = existing
elif existing:
    print(f"!! {path} is version {existing.get('version', 1)}; this notebook now writes "
          f"v{PREREG_VERSION}.")
    print("   Version 1 predates the code audit: its thresholds are on the raw score scale and")
    print("   it has no row for the rank-artifact hypothesis. If NO results exist yet, delete")
    print("   the file and re-run this cell. If results DO exist, keep v1 as the binding")
    print("   preregistration, record the v2 rows as exploratory, and say so in the write-up.")
    PREREGISTRATION = existing
else:
    with open(path, "w") as f:
        json.dump(PREREGISTRATION, f, indent=2)
    print(f"wrote {path} (v{PREREG_VERSION})")

print(f"\nwritten_at: {PREREGISTRATION['written_at']}")
groups = {}
for r in PREREGISTRATION["readings"]:
    groups.setdefault(r.get("group", "other"), []).append(r)
for g, rows in groups.items():
    print(f"\n{g}:")
    for r in rows:
        print(f"  {r['id']:>22}  {r['observation']}")


preregistration v2 already exists at /workspace/self_explainer/reports/preregistration.json — not overwriting.

written_at: 2026-08-18T01:15:10

primary:
               no_effect  Cfull curves for R-id and R-Q overlap at all N, both above the no-activation floor
       sample_efficiency  R-Q separated at low N, converging by the largest N
      attainable_optimum  R-Q strictly below R-id at all N, gap larger than the seed band
       collapse_to_floor  R-Q falls to the no-activation floor
             below_floor  R-Q falls below the no-activation floor
    collapse_to_baseline  R-Q collapses to the cross-family baseline (E_cross, P-rand)
         all_inside_band  Every arm sits inside the seed band
        metrics_disagree  The three metrics point in different directions

capacity:
       capacity_collapse  C0 and C128 collapse under R-Q while Cfull does not
              oracle_gap  Oracle >> Cfull/R-Q at the largest ladder N
       low_rank_recovery  Recovery threshold sits at low r

## 9. Summary — what NB00 decided

Read this before opening NB01.

In [11]:
print("PREFLIGHT SUMMARY")
print("=" * 72)
print(f"reuse from prior project  : NONE — the base repo's injection was a no-op, so every")
print(f"                            finished run was trained without the activation")
print(f"identity arm is free      : NO — Cfull/R-id is a fresh run like every other arm")
print(f"external checkouts needed : none — paper findings frozen in se/paper_audit.json")
print(f"paper's patching config   : C0 (no projector on either side, Appendix F.1)")
print(f"paper's projector rank    : {C.PAPER_LORA_R} (LoRA target module, Appendix F.3) — C128 arm")
print(f"our input map is LoRA-free: {'input_map' not in C.LORA_TARGET_MODULES} (asserted per run)")
print(f"target tied embeddings    : {target_cfg.tie_word_embeddings} (must be False)")
print(f"quantization              : {'4-bit — INVALID' if C.USE_4BIT else 'off (required)'}")
print(f"cached vector dim         : {vec_dim} == hidden_size {target_cfg.hidden_size}")
print(f"Cfull input-map params    : {budget.loc[budget.arm == 'Cfull'].iloc[0][f'params_x{C.N_LAYER_CHUNKS}_chunks']/1e6:.1f}M "
      f"({budget.loc[budget.arm == 'Cfull'].iloc[0]['pct_of_lora_budget']:.0f}% of LoRA budget)")
print(f"eval resolution           : +/- {binomial_half_width(0.5, C.EVAL_SIZE):.3f} at p=0.5, "
      f"n = {C.EVAL_SIZE}")
print(f"activation dynamic range  : {C.PAPER_WITH_ACTIVATION - C.PAPER_NO_ACTIVATION:.3f} "
      f"(paper Table 5) — all readings are normalized to the measured floor")
print(f"preregistration           : {C.REPORTS_DIR}/preregistration.json")
print()
print("EXIT CRITERIA (v2 section 4)")
print("  [x] reuse question settled  : the injection bug voids every finished patching")
print("      run. Only E_cross projector runs were ever candidates, and those are")
print("      superseded by the ridge arm. Every arm in this project is a fresh run.")
print("  [x] eval n and CI stated, against the ~4-point activation range")
print("  [x] target_modules question settled: the input map is trained whole, and every")
print("      run asserts its parameter count (revised section 4.1)")
print("  [ ] no-activation floor     : NB03 measures it at every N; it is the denominator")
print("      of every number this project reports")
print(f"  [{'x' if RUN_NOISE_FLOOR else ' '}] KL noise floor          : "
      + (f"see {C.REPORTS_DIR}/noise_floor.json" if RUN_NOISE_FLOOR
         else "not run on this CPU pod — measure it at the start of NB01"))
print()
print("BUDGET")
print("  The revised plan books 1.5 h for Phase 0 and adds three arms downstream: C0 under")
print("  both rotations, the no-activation floor across the sweep, and P-rand-full. The")
print("  eval set is 8x larger, so eval time is 8x. Count runs before starting NB03.")
print("  v2 books 19.5 h including a 2.0 h buffer. The injection bug spends that buffer:")
print("  Cfull/R-id is a fresh run either way, so the surprise is not the cost of the arm")
print("  but that the finished numbers measure text-only prediction and cannot anchor it.")
print("  Cut order if it runs long (v2 section 8), in order:")
print("    1. R-Qs distributional arm")
print("    2. intermediate ladder ranks C64/C512 — keep C0, Cfull, Oracle")
print("    3. Cfull-rand init control")
print("    4. Phase 1 on 8B: demonstrate invariance on Qwen3-0.6B only (see NB01)")
print("    5. extra Q seeds — NEVER before training seeds")
print("    6. low end of the N sweep")


PREFLIGHT SUMMARY
reuse from prior project  : NONE — the base repo's injection was a no-op, so every
                            finished run was trained without the activation
identity arm is free      : NO — Cfull/R-id is a fresh run like every other arm
external checkouts needed : none — paper findings frozen in se/paper_audit.json
paper's patching config   : C0 (no projector on either side, Appendix F.1)
paper's projector rank    : 128 (LoRA target module, Appendix F.3) — C128 arm
our input map is LoRA-free: True (asserted per run)
target tied embeddings    : False (must be False)
quantization              : off (required)
cached vector dim         : 4096 == hidden_size 4096
Cfull input-map params    : 67.1M (154% of LoRA budget)
eval resolution           : +/- 0.031 at p=0.5, n = 1024
activation dynamic range  : 0.041 (paper Table 5) — all readings are normalized to the measured floor
preregistration           : /workspace/self_explainer/reports/preregistration.json

EXIT CRITERIA